# 🚀 Hackathon de Ingeniería IA: Proyecto Integrador con Meta Llama
### Cuaderno Oficial de Construcción & Starter Kit para Participantes

Este cuaderno proporciona el entorno interactivo completo para que diseñes, construyas y pruebes tu solución de IA antes de pasar al Módulo 2. Integra **Model Routing**, **Búsqueda Vectorial RAG**, **Adaptadores LoRA** y **Microservicios con FastAPI**.

In [ ]:
# 1. INSTALACION DE LIBRERIAS ESENCIALES
!pip install transformers peft sentence-transformers fastapi uvicorn pydantic accelerate trl --quiet
print('✅ Dependencias del Hackathon instaladas con exito.')

In [ ]:
# 2. AUTENTICACION CON HUGGING FACE SECRETS
from google.colab import userdata
from huggingface_hub import login

try:
    login(token=userdata.get('HF_TOKEN'))
    print('✅ Sesion de Hugging Face iniciada correctamente.')
except Exception as e:
    print('⚠️ Aviso: No se encontro el secreto HF_TOKEN en Colab. Si usas modelos gated de Meta Llama, agregalo en el panel Secrets.')

## 🧩 Módulo 1: Enrutador Inteligente de Consultas (`ModelRouter`)
Evalúa la intención y longitud de la consulta del usuario para derivarla al canal de procesamiento óptimo.

In [ ]:
class ModelRouter:
    def __init__(self):
        self.keywords_rag = ['politica', 'reembolso', 'garantia', 'manual', 'horario', 'precio', 'envio', 'costo']
        self.keywords_lora = ['json', 'esquema', 'diagnostico', 'sql', 'codigo', 'formato']

    def route(self, query: str) -> str:
        text = query.lower()
        if any(k in text for k in self.keywords_rag):
            return 'RAG_PIPELINE'
        if any(k in text for k in self.keywords_lora):
            return 'LORA_ADAPTER'
        return 'FAST_LLM'

router = ModelRouter()
print('Prueba Router -> "Cual es la garantia?":', router.route('Cual es la garantia?'))
print('Prueba Router -> "Hola como estas?":', router.route('Hola como estas?'))

## 📚 Módulo 2: Motor RAG con Embeddings Normalizados (`VectorRAGEngine`)
Indexa la base de conocimiento de tu proyecto y busca por similitud coseno.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

class VectorRAGEngine:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.embedder = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None

    def index_documents(self, docs: list[str]):
        self.documents = docs
        self.embeddings = self.embedder.encode(docs, normalize_embeddings=True)

    def search(self, query: str, top_k=2, threshold=0.40):
        if self.embeddings is None or len(self.documents) == 0:
            return []
        q_emb = self.embedder.encode([query], normalize_embeddings=True)[0]
        scores = np.dot(self.embeddings, q_emb)
        indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in indices:
            if scores[idx] >= threshold:
                results.append({'text': self.documents[idx], 'score': float(scores[idx])})
        return results

# Base de datos documental de ejemplo para tu proyecto
docs_proyecto = [
    'La garantia de los dispositivos electronicos tiene una vigencia de 12 meses ante fallas de fabrica.',
    'Las devoluciones deben solicitarse dentro de los primeros 15 dias habiles con comprobante de compra.',
    'Los envios prioritarios se entregan en menos de 24 horas en zonas metropolitanas.'
]

rag = VectorRAGEngine()
rag.index_documents(docs_proyecto)
print('✅ Documentos indexados con exito. Vectores calculados en R^384.')
print('Busqueda de prueba:', rag.search('Cuanto tiempo tengo para devolver un producto?'))

## ⚡ Módulo 3: Pipeline Integrado de Inferencia
Conecta el Router con el RAG y la síntesis del modelo de lenguaje.

In [ ]:
import time

def responder_consulta_proyecto(consulta: str) -> dict:
    t0 = time.perf_counter()
    ruta = router.route(consulta)
    fuentes = []
    
    if ruta == 'RAG_PIPELINE':
        recuperados = rag.search(consulta, top_k=2, threshold=0.35)
        if recuperados:
            contexto = ' '.join([r['text'] for r in recuperados])
            respuesta = f'Segun nuestra documentacion oficial: {contexto}'
            fuentes = [r['text'] for r in recuperados]
        else:
            respuesta = 'No disponemos de informacion especifica en nuestros manuales para responder a esta consulta.'
    elif ruta == 'LORA_ADAPTER':
        respuesta = '{"estado": "procesado", "tipo": "consulta_especializada", "detalles": "Formato estructurado validado"}'
    else:
        respuesta = 'Hola, soy tu asistente inteligente de IA. ¿En que puedo orientarte hoy?'
        
    latencia = (time.perf_counter() - t0) * 1000
    return {
        'consulta': consulta,
        'ruta': ruta,
        'respuesta': respuesta,
        'latencia_ms': round(latencia, 2),
        'fuentes': fuentes
    }

# Probar consultas diversas
test_queries = [
    'Hola que tal',
    '¿Como es la politica para devoluciones?',
    'Genera un formato json con el estado'
]

for q in test_queries:
    res = responder_consulta_proyecto(q)
    print(f"\n[Query] {res['consulta']}")
    print(f"[Ruta]  {res['ruta']} ({res['latencia_ms']} ms)")
    print(f"[Resp]  {res['respuesta']}")

## 🌐 Módulo 4: Microservicio FastAPI con Validación Pydantic
Estructura el servidor web listo para producción.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title='Hackathon IA Engine', version='1.0.0')

class ChatRequest(BaseModel):
    mensaje: str
    usuario_id: str = 'usr_demo'

class ChatResponse(BaseModel):
    respuesta: str
    ruta_usada: str
    latencia_ms: float
    fuentes: list[str] = []

@app.post('/v1/chat', response_model=ChatResponse)
def chat_endpoint(req: ChatRequest):
    if not req.mensaje.strip():
        raise HTTPException(status_code=400, detail='Mensaje vacio.')
    res = responder_consulta_proyecto(req.mensaje)
    return ChatResponse(
        respuesta=res['respuesta'],
        ruta_usada=res['ruta'],
        latencia_ms=res['latencia_ms'],
        fuentes=res['fuentes']
    )

print('✅ Servidor FastAPI y modelos Pydantic definidos correctamente.')